In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Penggunaan beban kerja

<span id="usage"></span>
Penggunaan mewakili penggunaan perkhidmatan Qiskit Runtime dan ditentukan oleh jumlah masa QPU dikunci untuk melaksanakan beban kerja.

- Penggunaan Session diukur sebagai masa yang berlalu semasa sesi kekal aktif, kerana kapasiti QPU dikhaskan untuk tempoh sesi, tanpa mengira sama ada beban kerja sedang berjalan secara aktif. Lihat [Tempoh sesi](/guides/run-jobs-session#session-length) untuk maklumat lanjut tentang peralihan status sesi.
- Penggunaan Batch diukur sebagai jumlah kumulatif masa QPU dikunci untuk melaksanakan semua kerja dalam batch.
- Penggunaan kerja tunggal diukur sebagai masa QPU dikunci untuk melaksanakan kerja.

Perlu diingat bahawa kerja yang gagal atau dibatalkan dikira dalam penggunaan anda dalam keadaan tertentu - lihat bahagian [Kerja yang gagal dan dibatalkan](#failed-job) untuk butiran.

Bagi pengguna Pelan Pay-As-You-Go, lihat [Urus kos](/guides/manage-cost) untuk butiran tentang menetapkan had kos.

<span id="failed-job"></span>
## Penggunaan untuk kerja yang gagal dan dibatalkan
Apabila kerja gagal atau dibatalkan, penggunaan yang dilaporkan adalah seperti berikut:

* Mod kerja atau batch: Jika kegagalan atau pembatalan berlaku disebabkan ralat sistem, penggunaan yang dilaporkan adalah sifar. Bagi kerja yang gagal akibat ralat pengguna atau apabila pengguna membatalkan kerja, penggunaan yang dilaporkan adalah sebarang penggunaan yang telah berlaku sehingga masa itu, termasuk overhed yang ditanggung untuk menyediakan QPU bagi menjalankan kerja.

* Mod sesi: Penggunaan yang dilaporkan adalah masa jam dinding semasa sesi aktif, tanpa mengira bilangan kerja yang gagal atau dibatalkan.

<span id="view-usage"></span>
## Tanya penggunaan sebenar beban kerja
Selepas beban kerja selesai, terdapat beberapa cara untuk melihat penggunaan sebenarnya:

- Jalankan [`batch.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/batch#usage) atau [`session.usage()`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/session#usage) dalam `qiskit-ibm-runtime` 0.30 atau lebih baharu. Jika menggunakan versi lama `qiskit-ibm-runtime` (>= 0.23 dan < 0.30), penggunaan masih boleh didapati dalam `session.details()["usage_time"]` dan `batch.details()["usage_time"]`.
- Gunakan [`GET /sessions/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/sessions#tags__sessions__operations__GetSessionDetailsExtendedController_getSessionDetails) untuk melihat penggunaan bagi batch atau sesi tertentu.
- Gunakan [`GET /jobs/{id}`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/tags/jobs#tags__jobs__operations__GetJobByIdController_getJobById) untuk melihat penggunaan bagi kerja tunggal.

<span id="instance-usage"></span>
## Lihat penggunaan instans
Anda boleh melihat penggunaan instans pada halaman [Instans](https://quantum.cloud.ibm.com/instances), atau, bagi mereka yang mempunyai autoriti yang sesuai, halaman [Analitik](https://quantum.cloud.ibm.com/analytics). Perlu diingat bahawa halaman tersebut mungkin menunjukkan nombor penggunaan yang berbeza kerana ia mengira penggunaan secara berbeza.

Halaman Instans menunjukkan penggunaan masa nyata untuk 28 hari lepas (bergulir), sehingga masa semasa pada hari semasa. Penggunaan halaman Analitik dikira semula setiap jam dan merangkumi 28 hari penuh terakhir; iaitu, ia menunjukkan penggunaan dari 00:00 28 hari lalu hingga hari ini, pada jam teratas.

## Anggarkan penggunaan sebelum menghantar kerja
Walaupun mendapatkan anggaran setempat yang tepat adalah rumit kerana operasi tambahan yang dilakukan untuk penindasan dan mitigasi ralat, anda boleh menggunakan formula asas ini untuk mendapatkan anggaran penggunaan yang dianggarkan:

`<overhead setiap sub-kerja> + (rep_delay + <panjang litar>) * <bilangan pelaksanaan>`

- `<overhead setiap sub-kerja>` adalah overhead kira-kira 2 saat setiap sub-kerja. Ini termasuk operasi seperti memuatkan muatan ke dalam elektronik kawalan. Kerja primitif anda mungkin dibahagikan kepada berbilang sub-kerja jika terlalu besar untuk diproses oleh enjin pelaksanaan sekaligus.
- `rep_delay` adalah pilihan yang [boleh disesuaikan pengguna](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2#rep_delay), dan lalainya diberikan oleh `backend.default_rep_delay`, iaitu 250 mikrosaat pada kebanyakan backend IBM Quantum. Perlu diingat bahawa mengurangkan `rep_delay` mengurangkan jumlah masa pelaksanaan QPU, tetapi dengan kos kadar ralat penyediaan keadaan yang meningkat; lihat panduan [Pelaksanaan kadar pengulangan dinamik](/guides/repetition-rate-execution) untuk maklumat lanjut.
- `<panjang litar>` adalah jumlah panjang arahan. Setiap arahan mengambil masa yang berbeza pada QPU, jadi jumlah panjang berbeza-beza dari litar ke litar. Pengukuran, contohnya, boleh mengambil masa 56 kali lebih lama daripada gate `x`. `backend.target[<instruction>][<qubit>].duration` boleh digunakan untuk mencari tempoh tepat bagi setiap arahan. Panjang litar biasa mungkin antara 50-100 mikrosaat. Jika anda menggunakan teknik penindasan atau mitigasi ralat dengan primitif, arahan tambahan mungkin dimasukkan ke dalam litar anda, yang akan meningkatkan jumlah panjang litar.
    > **Note:** [Pilihan `scheduler_timing` eksperimen](/guides/visualize-circuit-timing) mengembalikan jumlah masa litar, tetapi ini BUKAN masa yang digunakan untuk pengebilan.
- `<bilangan pelaksanaan>` adalah jumlah bilangan litar darab bilangan shot, di mana litar adalah yang dijana selepas elemen PUB disiarkan.
    - Jika anda menggunakan teknik mitigasi ralat dengan primitif, litar tambahan mungkin dijalankan sebagai sebahagian daripada proses mitigasi, yang akan meningkatkan jumlah bilangan pelaksanaan. Selain itu, teknik mitigasi ralat lanjutan seperti PEA dan PEC datang dengan overhead yang jauh lebih tinggi kerana ia memerlukan menjalankan litar untuk pembelajaran hingar.
    - Estimator mengumpulkan boleh cerap yang berkomutat secara qubit-bijak, yang mengurangkan bilangan pelaksanaan.

Jika anda tidak menggunakan sebarang teknik mitigasi ralat lanjutan atau `rep_delay` tersuai, anda boleh menggunakan `2+0.00035*<bilangan pelaksanaan>` sebagai formula pantas.

### Anggarkan penggunaan secara setempat dengan Qiskit
Contoh kod ini menunjukkan cara menggunakan Qiskit untuk mengira masa litar: